In [0]:
from pyspark.sql import functions as F

# Read the three silver tables
df_persons = spark.table("assesment.silver.df_persons")
df_males = spark.table("assesment.silver.df_males")
df_females = spark.table("assesment.silver.df_females")

# Step 1: convert 90_and_over from text ("350,700") to integer (350700)
df_persons = df_persons.withColumn("90_and_over", F.regexp_replace("90_and_over", ",", "").cast("int"))
df_males = df_males.withColumn("90_and_over", F.regexp_replace("90_and_over", ",", "").cast("int"))
df_females = df_females.withColumn("90_and_over", F.regexp_replace("90_and_over", ",", "").cast("int"))

# Check
display(df_persons.select("population", "year", "90_and_over").orderBy("year"))
display(df_males.select("population", "year", "90_and_over").orderBy("year"))
display(df_females.select("population", "year", "90_and_over").orderBy("year"))

In [0]:
# Step 2: join df_females and df_males to df_persons on year
# All three have a column called 90_and_over, so rename each one first to avoid a clash
persons = df_persons.select("year", F.col("90_and_over").alias("90_and_over_num_persons"))
females = df_females.select("year", F.col("90_and_over").alias("90_and_over_num_females"))
males = df_males.select("year", F.col("90_and_over").alias("90_and_over_num_males"))

df_joined = (
    persons
    .join(females, on="year", how="inner")
    .join(males, on="year", how="inner")
    .orderBy("year")
)

print(f"Rows after join: {df_joined.count()}")   # expect 21
display(df_joined)

In [0]:
# Step 3: percentage of the 90_and_over population that is female / male, rounded to 2 decimal places
df_split = (
    df_joined
    .withColumn("90_and_over_pcnt_females",
                F.round(F.col("90_and_over_num_females") / F.col("90_and_over_num_persons") * 100, 2))
    .withColumn("90_and_over_pcnt_males",
                F.round(F.col("90_and_over_num_males") / F.col("90_and_over_num_persons") * 100, 2))
    .orderBy("year")
)

display(df_split)

In [0]:
# Step 4: display in the required column order, with percentages shown as e.g. 76.87%
df_result = df_split.select(
    "year",
    "90_and_over_num_persons",
    F.format_string("%.2f%%", "90_and_over_pcnt_females").alias("90_and_over_pcnt_females"),
    "90_and_over_num_females",
    F.format_string("%.2f%%", "90_and_over_pcnt_males").alias("90_and_over_pcnt_males"),
    "90_and_over_num_males",
).orderBy("year")

display(df_result)